# Pushshift Reddit Dataset: Collection + VADER Weak Labeling

This notebook builds the Pushshift-based Reddit dataset under the same unified project structure used for the Sentiment140 experiments.

It is designed to:
- collect Reddit comments from selected subreddits through the Pushshift-compatible API
- apply VADER weak labels for **negative / neutral / positive** sentiment
- save the final dataset under `Dataset/pushshift/`
- create fixed 60/10/30 train/validation/test splits under `Code/artifacts/pushshift/splits/`
- export a compact experiment configuration for later modelling notebooks

## 0. Environment Setup and Dependency Check

This section verifies the core packages required for data collection, weak labeling, and split generation. Running it first helps reduce environment-related errors when the notebook is moved to a new machine.

In [1]:
import importlib
import subprocess
import sys


def install_if_missing(package_name, import_name=None):
    module_name = import_name or package_name
    try:
        importlib.import_module(module_name)
        print(f"✅ {package_name} already installed")
    except ImportError:
        print(f"⬇️ Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


required_packages = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scikit-learn", "sklearn"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("nltk", "nltk"),
    ("requests", "requests"),
]

for package_name, import_name in required_packages:
    install_if_missing(package_name, import_name)

print(f"Python executable: {sys.executable}")

✅ numpy already installed
✅ pandas already installed
✅ scikit-learn already installed
✅ matplotlib already installed
✅ seaborn already installed
✅ nltk already installed
✅ requests already installed
Python executable: /opt/anaconda3/envs/happiness-nlp/bin/python


## 1. Align the Project Paths and Load VADER Resources

The notebook follows the same path logic as the Sentiment140 baseline so that datasets, artifacts, and reusable resources stay in predictable locations across the whole thesis project.

In [2]:
from pathlib import Path
import zipfile
import os
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

NOTEBOOK_DIR = Path.cwd()
CODE_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
PROJECT_ROOT = CODE_DIR.parent

DATASET_DIR = PROJECT_ROOT / "Dataset" / "pushshift"
ARTIFACT_DIR = CODE_DIR / "artifacts" / "pushshift"
CONFIG_DIR = ARTIFACT_DIR / "config"
MODEL_DIR = ARTIFACT_DIR / "models"
PRED_DIR = ARTIFACT_DIR / "predictions"
SPLIT_DIR = ARTIFACT_DIR / "splits"
RESOURCE_DIR = ARTIFACT_DIR / "resources"
NLTK_DIR = RESOURCE_DIR / "nltk_data"
SENTIMENT_DIR = NLTK_DIR / "sentiment"
VADER_DIR = SENTIMENT_DIR / "vader_lexicon"
VADER_ZIP_PATH = SENTIMENT_DIR / "vader_lexicon.zip"

for path in [DATASET_DIR, CONFIG_DIR, MODEL_DIR, PRED_DIR, SPLIT_DIR, VADER_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Clear proxy-related variables before requests.
for k in [
    "HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy",
    "ALL_PROXY", "all_proxy", "NO_PROXY", "no_proxy"
]:
    os.environ.pop(k, None)

# Prefer the project-local NLTK resource path shown in the thesis folder structure.
if str(NLTK_DIR.resolve()) not in nltk.data.path:
    nltk.data.path.insert(0, str(NLTK_DIR.resolve()))

# If the zipped VADER lexicon exists but is not extracted yet, unpack it locally.
if VADER_ZIP_PATH.exists() and not any(VADER_DIR.iterdir()):
    with zipfile.ZipFile(VADER_ZIP_PATH, "r") as zf:
        zf.extractall(VADER_DIR)

# Fallback: download the lexicon only if it still cannot be found locally.
try:
    nltk.data.find("sentiment/vader_lexicon.zip/vader_lexicon/vader_lexicon.txt")
except LookupError:
    try:
        nltk.data.find("sentiment/vader_lexicon/vader_lexicon.txt")
    except LookupError:
        nltk.download("vader_lexicon", download_dir=str(NLTK_DIR.resolve()))

sia = SentimentIntensityAnalyzer()

print("NOTEBOOK_DIR:", NOTEBOOK_DIR.resolve())
print("CODE_DIR:", CODE_DIR.resolve())
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("DATASET_DIR:", DATASET_DIR.resolve())
print("ARTIFACT_DIR:", ARTIFACT_DIR.resolve())
print("NLTK search path:", nltk.data.path[0])
print("✅ VADER loaded successfully")

NOTEBOOK_DIR: /Users/victor/Desktop/Graduation_Thesis/Code/notebooks
CODE_DIR: /Users/victor/Desktop/Graduation_Thesis/Code
PROJECT_ROOT: /Users/victor/Desktop/Graduation_Thesis
DATASET_DIR: /Users/victor/Desktop/Graduation_Thesis/Dataset/pushshift
ARTIFACT_DIR: /Users/victor/Desktop/Graduation_Thesis/Code/artifacts/pushshift
NLTK search path: /Users/victor/Desktop/Graduation_Thesis/Code/artifacts/pushshift/resources/nltk_data
✅ VADER loaded successfully


## 2. Build a Robust Requests Session

A reusable HTTP session with retry logic makes the collection process more stable and reduces failures caused by temporary network errors or rate limiting.

In [3]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def build_session():
    session = requests.Session()
    session.trust_env = False
    session.proxies = {}

    retry = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session


session = build_session()
print("✅ Session ready")

✅ Session ready


## 3. Define Collection Settings

The main collection parameters are kept in one place for easier adjustment and reproducibility. These values can be changed later without modifying the downstream processing logic.

In [4]:
import pandas as pd
from datetime import datetime, timezone
import time

SUBREDDITS = ["MadeMeSmile", "depression", "AskReddit"]
START_DATE = "2022-01-01"
END_DATE = "2022-12-31"
TARGET_PER_SUBREDDIT = 10000
PAGE_SIZE = 500
API = "https://api.pullpush.io/reddit/search/comment/"
RANDOM_STATE = 42

OUTPUT_DATA_PATH = DATASET_DIR / "reddit_vader_dataset.csv"
CONFIG_PATH = CONFIG_DIR / "pushshift_data_collection_config.json"

print("Output dataset path:", OUTPUT_DATA_PATH.resolve())

Output dataset path: /Users/victor/Desktop/Graduation_Thesis/Dataset/pushshift/reddit_vader_dataset.csv


## 4. Define Utility Functions

These helper functions handle timestamp conversion, basic text filtering, and VADER label assignment. The filtering is intentionally lightweight so that useful sentiment signals are not removed too early.

In [5]:
def to_unix(date_str: str) -> int:
    dt = datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    return int(dt.timestamp())


def is_valid(text: str) -> bool:
    if text in ["[deleted]", "[removed]"]:
        return False
    if text is None or len(text.strip()) < 5:
        return False
    if str(text).strip().startswith("http"):
        return False
    return True


def vader_label(text: str):
    score = sia.polarity_scores(text)["compound"]
    if score > 0.05:
        return 2, score      # positive
    elif score < -0.05:
        return 0, score      # negative
    else:
        return 1, score      # neutral

## 5. Fetch Reddit Comments and Apply Weak Labels

The main collection function requests Reddit comments page by page, filters invalid entries, applies VADER labeling, and stores a structured record for each retained comment.

In [6]:
def fetch(subreddit, start_ts, end_ts, target_n):
    results = []
    after = start_ts

    while len(results) < target_n:
        params = {
            "subreddit": subreddit,
            "after": after,
            "before": end_ts,
            "size": PAGE_SIZE,
            "sort": "asc",
        }

        try:
            r = session.get(API, params=params, timeout=30)
            r.raise_for_status()
            payload = r.json()

            if "data" not in payload:
                print("No data field returned:", payload)
                break

            data = payload["data"]
            if not data:
                break

            for item in data:
                text = item.get("body")
                if not is_valid(text):
                    continue

                label, score = vader_label(text)
                results.append(
                    {
                        "text": text,
                        "label": label,
                        "score": score,
                        "subreddit": subreddit,
                        "created_utc": item.get("created_utc"),
                    }
                )

                if len(results) >= target_n:
                    break

            after = data[-1]["created_utc"] + 1
            print(f"{subreddit}: {len(results)} collected")
            time.sleep(1)

        except Exception as e:
            print(f"Error in {subreddit}: {e}")
            break

    return results

## 6. Run Collection and Inspect the Dataset

This step executes the collection pipeline for each selected subreddit and merges the results into a single DataFrame for inspection.

In [9]:
start_ts = to_unix(START_DATE)
end_ts = to_unix(END_DATE)

all_data = []
for sr in SUBREDDITS:
    print(f"Fetching {sr} ...")
    data = fetch(sr, start_ts, end_ts, TARGET_PER_SUBREDDIT)
    all_data.extend(data)

df = pd.DataFrame(all_data)

if df.empty:
    raise ValueError("No data was collected. Please check the API endpoint, network connection, or date range.")

label_name_map = {0: "negative", 1: "neutral", 2: "positive"}
df["label_name"] = df["label"].map(label_name_map)

print("Dataset shape:", df.shape)
print("Label distribution:")
print(df["label"].value_counts().sort_index())
print("Subreddit distribution:")
print(df["subreddit"].value_counts())

df.head()

Fetching MadeMeSmile ...
MadeMeSmile: 94 collected
MadeMeSmile: 189 collected
MadeMeSmile: 277 collected
MadeMeSmile: 368 collected
MadeMeSmile: 457 collected
MadeMeSmile: 542 collected
MadeMeSmile: 620 collected
MadeMeSmile: 707 collected
MadeMeSmile: 792 collected
MadeMeSmile: 886 collected
MadeMeSmile: 975 collected
MadeMeSmile: 1048 collected
MadeMeSmile: 1141 collected
MadeMeSmile: 1234 collected
MadeMeSmile: 1322 collected
MadeMeSmile: 1412 collected
MadeMeSmile: 1503 collected
MadeMeSmile: 1598 collected
MadeMeSmile: 1694 collected
MadeMeSmile: 1787 collected
MadeMeSmile: 1881 collected
MadeMeSmile: 1960 collected
MadeMeSmile: 2039 collected
MadeMeSmile: 2126 collected
MadeMeSmile: 2212 collected
MadeMeSmile: 2309 collected
MadeMeSmile: 2399 collected
MadeMeSmile: 2473 collected
MadeMeSmile: 2552 collected
MadeMeSmile: 2639 collected
MadeMeSmile: 2725 collected
MadeMeSmile: 2811 collected
MadeMeSmile: 2892 collected
MadeMeSmile: 2983 collected
MadeMeSmile: 3077 collected
MadeMeS

,text,label,score,subreddit,created_utc,label_name
0,![gif](giphy|mFYTaY7Gth86xnE6N5),1,0.0000,MadeMeSmile,1640995203,neutral
1,Dayvon the boulder johnson.,1,0.0000,MadeMeSmile,1640995209,neutral
2,Congrats!! That's awesome!!,2,0.8647,MadeMeSmile,1640995213,positive
3,"As someone who has never been homeless, I can ...",0,-0.5423,MadeMeSmile,1640995213,negative
4,Congrats u f’ing legend!!!,2,0.6458,MadeMeSmile,1640995214,positive


## 7. Create Fixed 60/10/30 Stratified Splits

To stay consistent with the Sentiment140 pipeline, the weakly labeled Pushshift dataset is split into train, validation, and test subsets using the same 60/10/30 structure.

In [10]:
from sklearn.model_selection import train_test_split

split_source = df[["text", "label", "label_name", "score", "subreddit", "created_utc"]].copy()

train_df, temp_df = train_test_split(
    split_source,
    test_size=0.40,
    random_state=RANDOM_STATE,
    stratify=split_source["label"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.75,
    random_state=RANDOM_STATE,
    stratify=temp_df["label"],
)


def dist(name, frame):
    return {
        "split": name,
        "rows": len(frame),
        "negative": int((frame["label"] == 0).sum()),
        "neutral": int((frame["label"] == 1).sum()),
        "positive": int((frame["label"] == 2).sum()),
    }

for item in [dist("train", train_df), dist("val", val_df), dist("test", test_df)]:
    print(item)

{'split': 'train', 'rows': 18000, 'negative': 4380, 'neutral': 4381, 'positive': 9239}
{'split': 'val', 'rows': 3000, 'negative': 730, 'neutral': 730, 'positive': 1540}
{'split': 'test', 'rows': 9000, 'negative': 2190, 'neutral': 2190, 'positive': 4620}


## 8. Save the Dataset, Splits, and Configuration

The final dataset is saved under `Dataset/pushshift/`, while the fixed splits and experiment metadata are exported to `Code/artifacts/pushshift/` for later modelling notebooks.

In [11]:
import json

# Save the full collected dataset.
df.to_csv(OUTPUT_DATA_PATH, index=False, encoding="utf-8")

# Save fixed splits for downstream experiments.
split_frames = {
    "train_60.csv": train_df,
    "val_10.csv": val_df,
    "test_30.csv": test_df,
}

for filename, frame in split_frames.items():
    frame.to_csv(SPLIT_DIR / filename, index=False, encoding="utf-8")

config_payload = {
    "dataset": "pushshift_reddit_vader",
    "api": API,
    "subreddits": SUBREDDITS,
    "date_range": {"start": START_DATE, "end": END_DATE},
    "target_per_subreddit": TARGET_PER_SUBREDDIT,
    "page_size": PAGE_SIZE,
    "random_state": RANDOM_STATE,
    "split_ratio": {"train": 0.60, "val": 0.10, "test": 0.30},
    "label_mapping": {"0": "negative", "1": "neutral", "2": "positive"},
    "vader_thresholds": {"negative_lt": -0.05, "positive_gt": 0.05},
    "paths": {
        "dataset_csv": str(OUTPUT_DATA_PATH),
        "split_dir": str(SPLIT_DIR),
        "nltk_dir": str(NLTK_DIR),
    },
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config_payload, f, ensure_ascii=False, indent=2)

print("✅ Saved dataset to:", OUTPUT_DATA_PATH.resolve())
print("✅ Saved splits to:", SPLIT_DIR.resolve())
print("✅ Saved config to:", CONFIG_PATH.resolve())

✅ Saved dataset to: /Users/victor/Desktop/Graduation_Thesis/Dataset/pushshift/reddit_vader_dataset.csv
✅ Saved splits to: /Users/victor/Desktop/Graduation_Thesis/Code/artifacts/pushshift/splits
✅ Saved config to: /Users/victor/Desktop/Graduation_Thesis/Code/artifacts/pushshift/config/pushshift_data_collection_config.json


## 9. Preview the Saved Outputs

A final quick check confirms that the saved dataset and split files can be reloaded correctly.

In [12]:
saved_df = pd.read_csv(OUTPUT_DATA_PATH)
print(saved_df.shape)
saved_df.head()

(30000, 6)


,text,label,score,subreddit,created_utc,label_name
0,![gif](giphy|mFYTaY7Gth86xnE6N5),1,0.0000,MadeMeSmile,1640995203,neutral
1,Dayvon the boulder johnson.,1,0.0000,MadeMeSmile,1640995209,neutral
2,Congrats!! That's awesome!!,2,0.8647,MadeMeSmile,1640995213,positive
3,"As someone who has never been homeless, I can ...",0,-0.5423,MadeMeSmile,1640995213,negative
4,Congrats u f’ing legend!!!,2,0.6458,MadeMeSmile,1640995214,positive
